# Cleaning2
## Inizializzazione ed Import

In [2]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [3]:
file_codes = ['BAIPETNMRCFTP',
'AMPRION_ASYN_SAA',
'PLASMA_ABETA_PROJECT_ADX_VUMC',
'FNIH_PLASMA_PTAU_PROJECT',
'CSFALPHASYN',
'BLENNOWCSFNFL',
'UPENNPLASMA',
'BLENNOWPLASMANFL',
'BLENNOWPLASMATAU',
'C2N_PRECIVITYAD2_PLASMA',
'UGOTPTAU181',
'UPENN_PLASMA_FUJIREBIO_QUANTERIX',
'UCBERKELEY_AMY_6MM',
'UCBERKELEY_TAUPVC_6MM']

In [4]:
search = client.query_files(
    query={'custom.level' : 'cleaned_01', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


## Operazioni
- Eliminare i parametri con troppe poche righe
- Eliminare soffetti con solo 1 visita
- nuovi metadati (cofattori e fattori)

In [6]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned_BB'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned2BB'

if os.path.isfile(new_name+'.xlsx'):
    #aggiunge i filecode mancanti e riporta i file_code da riprocessare allo status precedente (variable names)
    update_new_support_file(support_file, new_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_name, rename_column=False)


new_support_file = pd.read_excel(new_name+'.xlsx')
dataCleaner = DataCleaner(support_file=new_support_file)

The ADNI_variables_cleaned2BB file has been updated with the new file_code: []
Open the file and verify it, if needed update the variables names and metadata
The ADNI_variables_cleaned2BB file has restored the previous information of the file_code: []
Open the file and verify it, if needed update the variables names and metadata


In [7]:
for file_name in zip_files.keys():
    print('\n\n ----', file_name)
    
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    new_support_file = dataCleaner.update_self_support_file(new_support_file)
    # Eliminare i parametri con troppe poche righesia dal DF che dal support file
    df_cleaned, file_code, new_support_file = dataCleaner.remove_param_few_subjects(df_new, file_name, prefix='cleaned/single_file')
    # --> funzione che trova i soggetti che hanno solo una visita quindi elimina quelle righe
    if file_code not in ['ADSP_PHC_BIOMARKER', 'PTDEMOG']:         # file solo con 1 visita, o info demog che anche una sola visita basta perchè baseline quindi da unire per ampliare il dataset ma non da usare da solo
        # Eliminare soggetti con solo 1 visita
        df_cleaned= dataCleaner.remove_sub_1visit(df_cleaned)
        # --> funzione che trova i parametri identificati da eliminare  ==> eliminare le colonne dal df

    # funzione che trasforma parametri categorici in dummies
    ref_list = ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']
    to_dummy_list = [x for x in ref_list if x in df_cleaned.columns]
    if to_dummy_list:
        final_df, bool_var = dataCleaner.classes_to_dummies(df_cleaned, col_list=to_dummy_list) 
    else:
        final_df = df_cleaned
    
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_02')

    # aggiunta di righe per i nuovi parametri e rimozione dal support di variabili non più nel df
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
    
    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_02', updated_support_file=new_support_file)    
    
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)
    
    # upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )

save_df(df_to_save=new_support_file, output_path=new_name) 




 ---- BAIPETNMRCFTP_08_17_22_11Aug2025_01.csv


 ---- AMPRION_ASYN_SAA_11Aug2025_01.csv
RID  non ha valori
usata scappatoia
EXAMDATE  non ha valori
usata scappatoia
VISCODE  non ha valori
usata scappatoia
VISIT_MONTH  non ha valori
usata scappatoia
Amprion_Result  non ha valori
usata scappatoia
METHOD  non ha valori
usata scappatoia
Npositive  non ha valori
usata scappatoia


 ---- PLASMA_ABETA_PROJECT_ADX_VUMC_11Aug2025_01.csv


 ---- FNIH_PLASMA_PTAU_PROJECT_11Aug2025_01.csv


 ---- CSFALPHASYN_03_21_14_11Aug2025_01.csv
VISIT_MONTH  non ha valori
usata scappatoia
RID  non ha valori
usata scappatoia
EXAMDATE  non ha valori
usata scappatoia
COHORT  non ha valori
usata scappatoia
ALPHA_SYN  non ha valori
usata scappatoia
METHOD  non ha valori
usata scappatoia
Npositive  non ha valori
usata scappatoia


 ---- BLENNOWCSFNFL_11Aug2025_01.csv
RID  non ha valori
usata scappatoia
VISCODE  non ha valori
usata scappatoia
VISIT_MONTH  non ha valori
usata scappatoia
EXAMDATE  non ha valori
usat

In [8]:
updated_metadata

{'file_code': 'UCBERKELEY_TAUPVC_6MM',
 'level': 'cleaned_02',
 'population': ['ADNI4', 'ADNI3', 'ADNI2'],
 'source': 'ADNI',
 'cofattori': [],
 'predittori': ['CSF_SUVR',
  'CTX_ENTORHINAL_VOLUME',
  'CSF_VOLUME',
  'CTX_FUSIFORM_VOLUME',
  'CTX_INFERIORPARIETAL_VOLUME',
  'CTX_INFERIORTEMPORAL_VOLUME',
  'CTX_LATERALOCCIPITAL_VOLUME',
  'CTX_MIDDLETEMPORAL_VOLUME',
  'CTX_PARAHIPPOCAMPAL_VOLUME',
  'CTX_PRECUNEUS_VOLUME'],
 'norm_scala': [],
 'norm_intervallo': ['CSF_SUVR',
  'CTX_ENTORHINAL_VOLUME',
  'CSF_VOLUME',
  'CTX_FUSIFORM_VOLUME',
  'CTX_INFERIORPARIETAL_VOLUME',
  'CTX_INFERIORTEMPORAL_VOLUME',
  'CTX_LATERALOCCIPITAL_VOLUME',
  'CTX_MIDDLETEMPORAL_VOLUME',
  'CTX_PARAHIPPOCAMPAL_VOLUME',
  'CTX_PRECUNEUS_VOLUME'],
 'norm_volume': [],
 'norm_scale_value': {'CSF_SUVR': {'FTP': [0.1678, 1.3657, 'increasing'],
   'unknown': [0.0125, 1.3087, 'increasing']}},
 'volume_norm_values': []}